# Классификация пользователей по возрастным категориям

Заказчик — IT-компания «Йети», управляющая сетью контекстной рекламы (РСЙ). РСЙ использует демографический таргетинг, поэтому ей нужна модель, которая по «цифровому следу» анонимного пользователя (логам посещений, активности с рекламой, глубине серфинга, использованию облака) определяет возрастную категорию.

**Задача в терминах ML.** Многоклассовая классификация с 5 классами:

| Метка | Возраст        |
|------:|----------------|
| 0     | младше 18 лет  |
| 1     | 18–25 лет      |
| 2     | 26–40 лет      |
| 3     | 41–55 лет      |
| 4     | 56+ лет        |

**Целевая метрика.** Макро-F1, поскольку модель должна одинаково хорошо различать все классы, даже редкие. Пороговое значение для рекомендации модели к внедрению: **F1_macro ≥ 0.75** одновременно на кросс-валидации и на отложенной тестовой выборке. Вспомогательные метрики — `precision_macro`, `recall_macro`.

**Бизнес-ограничение.** Показы рекламы 18+ несовершеннолетним влекут штрафы и репутационные риски, поэтому при равном F1 предпочтительнее модели, реже завышающие возраст для класса 0 (младше 18).

**План работы.**

1. Подготовка среды и библиотек.
2. Исследовательский анализ данных (EDA).
3. Предобработка данных и формирование признаков.
4. Обучение и оценка базовой модели (`DummyClassifier`).
5. Создание и отбор признаков.
6. Подбор гиперпараметров `LogisticRegression` и `SVC` с разными ядрами.
7. Подготовка артефактов модели для внедрения.
8. Итоговые выводы.

## Подготовка среды и библиотек

В этом разделе фиксируем версии библиотек, импортируем их одним блоком, объявляем константы (`RANDOM_STATE`, `TEST_SIZE`, пути) и загружаем сырые датафреймы. Загрузка построена с обработкой `try/except`: сначала пробуем путь Практикума `/datasets/`, при ошибке — локальную папку `SPRINT_13/input/`. Это позволит ревьюеру выполнить ноутбук без правки кода.

### Установка библиотек

Список зависимостей, на которых проверялся ноутбук. На платформе Практикума они уже установлены — ячейка не нужна, но при локальном запуске её можно раскомментировать.

In [ ]:
# Список зависимостей сохраняется в requirements.txt прямо в проект,
# чтобы окружение можно было поднять одной командой. Версии зафиксированы под
# среду JupyterHub Практикума (Python 3.9 + sklearn 0.24) — так тетрадь
# гарантированно прогоняется и у нас, и у ревьюера без адаптеров под API.
requirements = """\
pandas==1.2.5
numpy==1.19.5
scikit-learn==0.24.2
matplotlib==3.4.3
seaborn==0.11.2
statsmodels==0.12.2
joblib==1.0.1
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

# Установку библиотек удобно выполнять одной строкой; при работе
# на платформе Практикума зависимости уже установлены — раскомментируйте при локальном запуске.
# !pip install -r requirements.txt

### Импорты

Все импорты собраны в одной ячейке и сгруппированы по PEP 8: стандартная библиотека → научный стек → визуализация → ML.

In [ ]:
# --- стандартная библиотека ---
import warnings
from pathlib import Path

# --- научный стек ---
import numpy as np
import pandas as pd

# --- визуализация ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- ML (sklearn 0.24) ---
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

# --- утилиты ---
import joblib
from IPython.display import display
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

### Константы

Константы вынесены в отдельную ячейку, чтобы их было легко найти и переиспользовать.

In [ ]:
RANDOM_STATE = 42      # фиксирует все стохастические алгоритмы для воспроизводимости
TEST_SIZE = 0.25       # ~25% уникальных пользователей идут в отложенный тест (требование задачи 20–30%)
N_SPLITS = 5           # фолды для стратифицированной кросс-валидации
N_JOBS = -1            # параллелизм grid search

# Имена возрастных категорий — для подписей графиков и таблиц
AGE_LABELS = {
    0: "<18",
    1: "18–25",
    2: "26–40",
    3: "41–55",
    4: "56+",
}

# Путь к данным: платформа Практикума хранит файлы в /datasets,
# локально мы храним их рядом с тетрадью.
DATA_PATHS = ["/datasets/", "SPRINT_13/input/", "input/", "./"]

# Куда сохраняем артефакты для внедрения.
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

# --- Артефактный модуль: единый источник истины для ManualStandardScaler и build_features ---
# Файл `feature_pipeline.py` лежит рядом с тетрадью (см. репозиторий проекта). И сама
# тетрадь, и сохранённая модель импортируют классы из него — поэтому joblib.dump/load
# работает без хаков.
import sys
sys.path.insert(0, str(Path.cwd()))   # на случай, если cwd не на sys.path (некоторые ядра)

from feature_pipeline import (
    ManualStandardScaler,
    build_features,
    DAYTIME_ORDER,
    DAYTIME_DROP,
    ALL_CATEGORIES,
    CATEGORY_DROP,
)
print(f"feature_pipeline загружен из {Path('feature_pipeline.py').resolve()}")

### Загрузка датафреймов

Функция `load_csv` перебирает кандидатские каталоги; первый, в котором лежит нужный файл, используется как источник. Так один и тот же ноутбук без правок работает и на Практикуме, и локально.

In [ ]:
def load_csv(filename: str, candidates=DATA_PATHS) -> pd.DataFrame:
    """Возвращает датафрейм, пробуя несколько каталогов по очереди."""
    last_err = None
    for base in candidates:
        path = Path(base) / filename
        try:
            return pd.read_csv(path)
        except FileNotFoundError as err:
            last_err = err
    raise FileNotFoundError(
        f"Файл {filename} не найден ни в одном из каталогов: {candidates}"
    ) from last_err


users_raw    = load_csv("ds_s13_users.csv")
visits_raw   = load_csv("ds_s13_visits.csv")
ads_raw      = load_csv("ads_activity.csv")
surf_raw     = load_csv("surf_depth.csv")
cloud_raw    = load_csv("cloud_usage.csv")
primary_raw  = load_csv("primary_device.csv")

print(
    f"users={users_raw.shape}, visits={visits_raw.shape}, ads={ads_raw.shape}, "
    f"surf={surf_raw.shape}, cloud={cloud_raw.shape}, primary={primary_raw.shape}"
)

## Исследовательский анализ данных

Цели EDA на этом проекте:

- понять структуру каждого источника и сопоставить его с описанием в ТЗ;
- найти пропуски, дубликаты и аномалии и выбрать стратегии их обработки;
- проверить пересечение пользователей между таблицами — мы планируем объединять их по `user_id`;
- сравнить поведение пользователей разных возрастных категорий, чтобы убедиться, что в данных есть полезный сигнал.

Все DataFrame выводятся через `display()`; крупные блоки текста заменяем компактными таблицами. Каждый график снабжён заголовком и подписями осей.

### Краткая сводка по каждому источнику

Сначала смотрим размерности, типы, число уникальных пользователей и количество дубликатов по `user_id`. Сводим всё в одну таблицу — её удобнее читать, чем серию выводов `info()`.

In [ ]:
SOURCES = {
    "users":   users_raw,
    "visits":  visits_raw,
    "ads":     ads_raw,
    "surf":    surf_raw,
    "cloud":   cloud_raw,
    "primary": primary_raw,
}

summary = pd.DataFrame(
    {
        name: {
            "rows": df.shape[0],
            "cols": df.shape[1],
            "unique_users": df["user_id"].nunique(),
            "dup_user_ids": int(df["user_id"].duplicated().sum()),
            "missing_total": int(df.isna().sum().sum()),
        }
        for name, df in SOURCES.items()
    }
).T
display(summary)

**Что видно из сводки.**

- В `users` 5913 строк при 5826 уникальных идентификаторах — 87 строк-дубликатов. То же самое в `ads` (5826 строк, 5593 уникальных пользователя, 233 дубля).
- В `surf`, `cloud` и `primary_device` дублей нет, но количество строк меньше, чем уникальных пользователей в `users` — значит, эти признаки покрывают не всех.
- Явных пропусков (NaN) в столбцах нет — все «пропуски» возникнут на стадии слияния как отсутствующие строки.
- Покрытие `primary_device` — 5669 пользователей из 5826 (97.3%): 157 без значения, кодируем как `unknown` при сборке признаков.

Дальше проверим, согласуются ли дублирующиеся записи между собой.

In [ ]:
# Дубликаты могут быть «безопасными» (повторяет ту же запись) или конфликтующими.
# Если у одного user_id два разных значения целевого признака — это уже проблема.
def check_dup_consistency(df, key, value):
    dup_ids = df.loc[df[key].duplicated(keep=False), key].unique()
    conflict = (
        df[df[key].isin(dup_ids)]
        .groupby(key)[value]
        .nunique()
        .gt(1)
        .sum()
    )
    return len(dup_ids), int(conflict)


checks = pd.DataFrame(
    [
        ("users", *check_dup_consistency(users_raw, "user_id", "age_category")),
        ("ads",   *check_dup_consistency(ads_raw,   "user_id", "ads_activity")),
    ],
    columns=["table", "users_with_dups", "users_with_conflicting_dups"],
).set_index("table")
display(checks)

Конфликтов нет — все дубликаты повторяют одну и ту же запись. Значит, на этапе предобработки достаточно `drop_duplicates(subset='user_id')` без потери информации.

### Целевой признак

Смотрим распределение классов: оно ожидаемо несбалансировано, поэтому мы:

- стратифицируем train/test и фолды по `age_category`;
- усредняем метрики качества по классам (`macro`);
- в `LogisticRegression`/`SVC` будем использовать `class_weight="balanced"`, чтобы не получить модель, всегда предсказывающую большинство.

In [ ]:
users_unique = users_raw.drop_duplicates(subset="user_id").reset_index(drop=True)

target_dist = (
    users_unique["age_category"]
    .value_counts()
    .sort_index()
    .rename_axis("age_category")
    .to_frame("users")
    .assign(share=lambda d: (d["users"] / d["users"].sum()).round(4))
)
target_dist.index = target_dist.index.map(AGE_LABELS)
display(target_dist)

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    x=target_dist.index, y=target_dist["users"], ax=ax, color="#4C72B0"
)
ax.set_title("Распределение возрастных категорий (всего {} пользователей)".format(
    users_unique.shape[0]
))
ax.set_xlabel("Возрастная категория")
ax.set_ylabel("Число пользователей")
for p, share in zip(ax.patches, target_dist["share"]):
    ax.annotate(
        f"{share:.0%}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center", va="bottom", fontsize=10,
    )
fig.tight_layout()
plt.show()

Самая крупная группа — `56+` (≈30.8% пользователей), самая редкая — `18–25` (≈8.9%). Отношение крупнейшего к меньшему классу ≈ 3.4×: дисбаланс умеренный, но без коррекции базовая модель будет «съезжать» к классу 4.

### Лог посещений

В логе ~1.07 млн событий — это наш главный источник поведенческих признаков. Проверим даты, время суток и категории сайтов.

In [ ]:
visits_raw["date"] = pd.to_datetime(visits_raw["date"])

display(visits_raw.head(3))

visits_overview = pd.Series(
    {
        "events": visits_raw.shape[0],
        "unique_users": visits_raw["user_id"].nunique(),
        "unique_sessions": visits_raw["session_id"].nunique(),
        "unique_categories": visits_raw["website_category"].nunique(),
        "date_min": visits_raw["date"].min().date(),
        "date_max": visits_raw["date"].max().date(),
        "days": visits_raw["date"].nunique(),
    },
    name="visits",
)
display(visits_overview.to_frame())

In [ ]:
# Активность пользователей: сколько сессий и событий приходится на одного.
per_user = (
    visits_raw.groupby("user_id")
    .agg(events=("session_id", "size"), sessions=("session_id", "nunique"),
         days=("date", "nunique"))
)
display(per_user.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.histplot(per_user["sessions"], bins=40, ax=axes[0], color="#4C72B0")
axes[0].set_title("Сессий на пользователя за 2 недели")
axes[0].set_xlabel("Сессии")
axes[0].set_ylabel("Пользователи")

sns.histplot(per_user["events"], bins=40, ax=axes[1], color="#55A868")
axes[1].set_title("Событий на пользователя за 2 недели")
axes[1].set_xlabel("События")
axes[1].set_ylabel("Пользователи")
fig.tight_layout()
plt.show()

# Боксплоты по активности — отдельно для оценки выбросов (квартили + усы 1.5 IQR).
fig, axes = plt.subplots(1, 2, figsize=(11, 2.6))
sns.boxplot(x=per_user["sessions"], ax=axes[0], color="#4C72B0")
axes[0].set_title("Боксплот: сессии на пользователя")
axes[0].set_xlabel("Сессии")
sns.boxplot(x=per_user["events"], ax=axes[1], color="#55A868")
axes[1].set_title("Боксплот: события на пользователя")
axes[1].set_xlabel("События")
fig.tight_layout()
plt.show()

# Численная оценка «выбросов»: правый ус IQR и сколько пользователей за ним.
def iqr_outlier_summary(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    return pd.Series({
        "p25": round(q1, 1), "p75": round(q3, 1),
        "верхний ус": round(upper, 1),
        "за усом": int((s > upper).sum()),
        "max": int(s.max()),
    })

display(pd.DataFrame({
    "events":   iqr_outlier_summary(per_user["events"]),
    "sessions": iqr_outlier_summary(per_user["sessions"]),
    "days":     iqr_outlier_summary(per_user["days"]),
}))

In [ ]:
# Время суток (тоже категория) и категории сайтов — насколько они «общие».
daytime_dist = visits_raw["daytime"].value_counts(normalize=True).round(4)
display(daytime_dist.to_frame("share"))

cat_top = (
    visits_raw["website_category"].value_counts(normalize=True).head(10).round(4)
)
display(cat_top.to_frame("share"))

**Что важно по логу.**

- Лог покрывает 14 дней (с 1 по 14 ноября 2025). Минимальный пользователь имеет ≥ 100 событий — «однораза» нет.
- В среднем 183 события и 180 сессий на пользователя; распределения сильно скошены вправо. Боксплот показывает несколько сотен пользователей «за усом» (≥ p75 + 1.5·IQR), но это не аномалии — это активные пользователи. Удалять их не надо: они увеличивают сигнал для класса «активные взрослые».
- **`n_active_days` фактически бинарный**: 13 или 14, со средним 14.00 и `std ≈ 0.04`. Сам по себе он почти не несёт информации, поэтому в `build_features` он входит только как знаменатель `sessions_per_day` (среднее число сессий в день).
- День (37%) и вечер (36%) доминируют, ночь — всего 8%. Это намекает на сильный временной сигнал по возрастам.
- 20 категорий сайтов — компактное пространство, можно безопасно перевести в pivot.

### Поведенческие признаки vs возраст

Перед моделированием полезно увидеть, что сигнал действительно есть. Сравним долю активности по времени суток в разрезе возрастной категории и тепловую карту «возраст × категория сайта».

In [ ]:
visits_with_age = visits_raw.merge(users_unique, on="user_id", how="left")

# 1) Доля активности по времени суток для каждой возрастной категории.
daytime_age = (
    pd.crosstab(
        visits_with_age["age_category"], visits_with_age["daytime"], normalize="index"
    )[["утро", "день", "вечер", "ночь"]]
    .rename(index=AGE_LABELS)
)
display(daytime_age.round(3))

fig, ax = plt.subplots(figsize=(8, 3.6))
daytime_age.plot(kind="bar", stacked=True, ax=ax, colormap="viridis")
ax.set_title("Доля активности по времени суток внутри каждой возрастной категории")
ax.set_xlabel("Возрастная категория")
ax.set_ylabel("Доля событий")
ax.legend(title="Время суток", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
fig.tight_layout()
plt.show()

In [ ]:
# 2) Сводная карта «возраст × категория сайта» — доля посещений каждой категории
#    внутри возрастной группы. Так видны категории-индикаторы.
cat_age = (
    pd.crosstab(
        visits_with_age["age_category"], visits_with_age["website_category"],
        normalize="index",
    )
    .rename(index=AGE_LABELS)
)

fig, ax = plt.subplots(figsize=(13, 3.2))
sns.heatmap(cat_age, cmap="viridis", annot=False, cbar_kws={"label": "Доля посещений"}, ax=ax)
ax.set_title("Доля посещений категорий сайтов внутри каждой возрастной группы")
ax.set_xlabel("Категория сайта")
ax.set_ylabel("Возрастная категория")
ax.tick_params(axis="x", labelrotation=45, labelsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# Численно подсчитаем разницу долей по категориям между крайними возрастными группами:
# это первичная оценка «силы» категориальных признаков.
contrast = (cat_age.max(axis=0) - cat_age.min(axis=0)).sort_values(ascending=False).round(4)
display(contrast.head(10).to_frame("max−min доля по возрастам"))

Колонка «max−min» показывает амплитуду доли каждой категории между возрастными группами. Категории с самой высокой амплитудой — самые «разделяющие». Это уже подсказка для отбора признаков: даже без модели видно, что у разных возрастов разный профиль предпочтений.

### Маленькие справочные таблицы: ads, surf, cloud

Эти источники не покрывают всех пользователей. При мердже у тех, кто отсутствует, появится пропуск — заполним его явным маркером `unknown`. Сам факт пропуска тоже может коррелировать с возрастом.

In [ ]:
ads_unique     = ads_raw.drop_duplicates(subset="user_id").reset_index(drop=True)
surf_unique    = surf_raw.copy()
cloud_unique   = cloud_raw.copy()
primary_unique = primary_raw.copy()

coverage = pd.DataFrame({
    "ads":     {"users": ads_unique["user_id"].nunique(),
                "missing_vs_users": users_unique["user_id"].nunique() - ads_unique["user_id"].nunique()},
    "surf":    {"users": surf_unique["user_id"].nunique(),
                "missing_vs_users": users_unique["user_id"].nunique() - surf_unique["user_id"].nunique()},
    "cloud":   {"users": cloud_unique["user_id"].nunique(),
                "missing_vs_users": users_unique["user_id"].nunique() - cloud_unique["user_id"].nunique()},
    "primary": {"users": primary_unique["user_id"].nunique(),
                "missing_vs_users": users_unique["user_id"].nunique() - primary_unique["user_id"].nunique()},
}).T
display(coverage)

ads_dist     = ads_unique["ads_activity"].value_counts(normalize=True).round(3).to_frame("share")
surf_dist    = surf_unique["surf_depth"].value_counts(normalize=True).round(3).to_frame("share")
cloud_dist   = cloud_unique["cloud_usage"].value_counts(normalize=True).round(3).to_frame("share")
primary_dist = primary_unique["primary_device"].value_counts(normalize=True).round(3).to_frame("share")
display(ads_dist, surf_dist, cloud_dist, primary_dist)

In [ ]:
# Сразу посмотрим связь ads_activity / surf_depth / cloud_usage / primary_device с возрастной категорией.
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))

for ax, source, col in [
    (axes[0], ads_unique,     "ads_activity"),
    (axes[1], surf_unique,    "surf_depth"),
    (axes[2], cloud_unique,   "cloud_usage"),
    (axes[3], primary_unique, "primary_device"),
]:
    cross = (
        pd.crosstab(
            users_unique.set_index("user_id").join(source.set_index("user_id"))["age_category"],
            users_unique.set_index("user_id").join(source.set_index("user_id"))[col],
            normalize="index",
        )
        .rename(index=AGE_LABELS)
    )
    cross.plot(kind="bar", stacked=True, ax=ax, colormap="viridis", legend=False)
    ax.set_title(f"{col} по возрастам")
    ax.set_xlabel("Возрастная категория")
    ax.set_ylabel("Доля")
    ax.tick_params(axis="x", labelrotation=0)
    ax.legend(title=col, fontsize=8, loc="upper right")

fig.tight_layout()
plt.show()

### Промежуточные выводы по EDA

- **Объём.** 5826 уникальных пользователей, у каждого ≥ 100 событий лога за 14 дней; данные не «полупустые».
- **Дубликаты.** В `users` и `ads` встречаются полностью повторяющиеся строки по `user_id` (87 и 233 соответственно). Конфликтов в значениях нет — безопасно убрать через `drop_duplicates`.
- **Пропуски.** Явных NaN нет. Однако 233 пользователя без `ads_activity`, 111 без `surf_depth`, 146 без `cloud_usage`, 157 без `primary_device` — заполним маркером `unknown`.
- **Целевой признак.** Несбалансирован (минимум 8.9% у класса 18–25, максимум 30.8% у 56+). Решения: `class_weight="balanced"`, стратификация, macro-метрики.
- **Поведенческий сигнал.** Доли категорий сайтов и распределение по времени суток заметно различаются между возрастными группами, что внушает оптимизм для линейных и SVM-моделей с правильным набором признаков.
- **Источники между собой.** Все пользователи из `users` есть в `visits` — лог посещений станет «осью» признакового пространства; остальные таблицы (`ads`, `surf`, `cloud`, `primary_device`) присоединяются left-join.

## Предобработка данных

Все «жёсткие» операции по чистке (дедупликация, типизация) выносим сразу. Затем оформляем функцию `build_features`, которая по сырым датафреймам возвращает готовое признаковое пространство. Эта же функция понадобится при внедрении модели в эксплуатацию — поэтому её мы сохраним отдельно.

Что внутри функции:

1. **Объём активности** — `n_events` (всего событий) и `sessions_per_day` = `n_sessions / n_active_days` (среднее число сессий в день — прямой пункт ТЗ). Сами `n_sessions` и `n_active_days` после расчёта отношения убираем, чтобы не получить триаду `A, B, A/B`, на которую ревьюер ранее указывал отдельно.
2. **Поведенческие профили** — доля посещений по времени суток (4 колонки → 3 после удаления `ночь`, чтобы сумма перестала быть равной 1) и доля по 20 категориям сайтов (19 после удаления `Category 20` по той же причине).
3. **Категориальные признаки**:
    - `ads_activity`, `surf_depth`, `cloud_usage`, `primary_device` — справочники, присоединяются left-join;
    - `peak_daytime` — время суток, на которое приходится максимум активности пользователя (прямой пункт ТЗ «наиболее активное время суток»). Пропуски справочников заполняются явным маркером `unknown`.
4. Итоговый датафрейм индексирован по `user_id`.

Кодирование (OHE) и масштабирование выполняются на следующем шаге уже в `Pipeline`, чтобы статистики считались только по train и не было утечки.

### Чистка исходных таблиц

Сохраним очищенные версии: убираем точные дубликаты по `user_id`, типизируем дату, переводим `cloud_usage` в строку (чтобы `unknown` сосуществовал с `True/False`).

In [ ]:
visits_clean = visits_raw.copy()
visits_clean["date"] = pd.to_datetime(visits_clean["date"])

users_clean    = users_raw.drop_duplicates(subset="user_id").reset_index(drop=True)
ads_clean      = ads_raw.drop_duplicates(subset="user_id").reset_index(drop=True)
surf_clean     = surf_raw.drop_duplicates(subset="user_id").reset_index(drop=True)
cloud_clean    = cloud_raw.drop_duplicates(subset="user_id").reset_index(drop=True)
cloud_clean["cloud_usage"] = cloud_clean["cloud_usage"].astype(str)
primary_clean  = primary_raw.drop_duplicates(subset="user_id").reset_index(drop=True)

shape_log = pd.DataFrame({
    "users":   [users_raw.shape[0],   users_clean.shape[0]],
    "ads":     [ads_raw.shape[0],     ads_clean.shape[0]],
    "surf":    [surf_raw.shape[0],    surf_clean.shape[0]],
    "cloud":   [cloud_raw.shape[0],   cloud_clean.shape[0]],
    "primary": [primary_raw.shape[0], primary_clean.shape[0]],
}, index=["до", "после dedup"])
display(shape_log)

### Функция формирования признаков

Функция `build_features` живёт в отдельном файле [`feature_pipeline.py`](feature_pipeline.py) рядом с тетрадью — как и просит ТЗ. И тетрадь, и сохранённая модель импортируют её оттуда же, поэтому `joblib.dump`/`joblib.load` корректно сериализуют `ManualStandardScaler` без monkey-патчей. Списки колонок для отбрасывания (`DAYTIME_DROP`, `CATEGORY_DROP`) и фиксированный порядок временных интервалов — тоже константы модуля. `peak_daytime` считается через `idxmax` по таблице долей.

In [ ]:
# build_features уже импортирована из artifacts/feature_pipeline.py — это единый
# источник истины и для тетради, и для модели в продакшене.
# Здесь только применяем её к очищенным датафреймам и смотрим результат.
X_full = build_features(visits_clean, ads_clean, surf_clean, cloud_clean, primary_clean)
print(f"X_full: {X_full.shape[0]} пользователей, {X_full.shape[1]} признаков")
display(X_full.head())

# Покажем исходник функции, чтобы её было удобно читать прямо в тетради:
import inspect
print("\n--- artifacts/feature_pipeline.py :: build_features ---")
print(inspect.getsource(build_features))

Получили 29 признаков: 24 числовых (1 объёмный `n_events`, 1 ставка `sessions_per_day`, 3 доли времени суток без `ночь`, 19 долей категорий сайтов без `Category 20`) и 5 категориальных (`ads_activity`, `surf_depth`, `cloud_usage`, `primary_device`, `peak_daytime`). Колонки `daytime_ночь` и `cat_Category 20` отброшены, чтобы убрать математическую зависимость от остальных долей; `n_sessions` и `n_active_days` свернули в `sessions_per_day` по правилу «A, B, A/B → оставляй только что-то одно».

### Разделение на train / test

Согласно требованию, тестовая выборка — 20–30% **уникальных пользователей**. Выбираем 25% и стратифицируем по `age_category`. Все последующие операции (масштабирование, OHE, отбор признаков, подбор гиперпараметров) делаем только на train.

In [ ]:
y_full = users_clean.set_index("user_id").loc[X_full.index, "age_category"].astype(int)
assert (X_full.index == y_full.index).all(), "Индексы X и y должны совпадать"

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=TEST_SIZE,
    stratify=y_full,
    random_state=RANDOM_STATE,
)

# Проверка: одни пользователи в test, другие в train — пересечения нет.
overlap = set(X_train.index).intersection(set(X_test.index))
print(f"train: {X_train.shape[0]}, test: {X_test.shape[0]}, пересечений: {len(overlap)}")

split_dist = pd.concat(
    [
        y_train.value_counts(normalize=True).rename("train"),
        y_test.value_counts(normalize=True).rename("test"),
    ],
    axis=1,
).sort_index().rename(index=AGE_LABELS).round(3)
display(split_dist)

### Препроцессор: ручной стандартизатор и OHE без утечки

`ManualStandardScaler` (определён в [`feature_pipeline.py`](feature_pipeline.py)) реализует стандартизацию вручную — через `(x - mean) / std`, как просили ревьюеры. Параметры считаются только в `fit`, поэтому при использовании внутри `Pipeline` статистики возьмутся только из train-фолда и утечки не будет. Категориальные признаки кодируем `OneHotEncoder(handle_unknown='ignore')`, чтобы новые значения в тесте не ломали трансформ.

`ColumnTransformer` собирает оба блока, а `make_preprocessor()` возвращает свежий экземпляр — это пригодится при перестройке пайплайнов с разными списками признаков.

In [ ]:
# ManualStandardScaler импортирован из artifacts/feature_pipeline.py (см. начало тетради).
# Здесь только конфигурируем препроцессор: какие колонки куда.

CATEGORICAL_COLS = ["ads_activity", "surf_depth", "cloud_usage", "primary_device", "peak_daytime"]
NUMERIC_COLS = [c for c in X_full.columns if c not in CATEGORICAL_COLS]


def make_preprocessor(numeric_cols=NUMERIC_COLS, categorical_cols=CATEGORICAL_COLS):
    return ColumnTransformer(
        transformers=[
            ("num", ManualStandardScaler(), numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse=False), categorical_cols),
        ],
        remainder="drop",
    )


print(f"числовых признаков: {len(NUMERIC_COLS)}")
print(f"категориальных признаков: {len(CATEGORICAL_COLS)}")

In [ ]:
# Санити-чек препроцессора: проверяем, что fit/transform отрабатывают и числовые
# колонки приходят с mean≈0, std≈1 на train. Делаем это на ПОЛНОМ NUMERIC_COLS (24 кол.),
# до отбора по VIF — финальная модель ниже будет использовать NUMERIC_FINAL.
_preprocessor = make_preprocessor()
_preprocessor.fit(X_train)
_X_train_t = _preprocessor.transform(X_train)
_X_test_t  = _preprocessor.transform(X_test)
print(f"X_train после препроцессинга (до VIF-фильтра): {_X_train_t.shape}")
print(f"X_test  после препроцессинга (до VIF-фильтра): {_X_test_t.shape}")
# Проверка: mean ≈ 0, std ≈ 1 на train (по числовым колонкам)
num_part = _X_train_t[:, :len(NUMERIC_COLS)]
print(f"mean числовых на train: {num_part.mean(axis=0).round(3)[:5]} ...")
print(f"std  числовых на train: {num_part.std(axis=0).round(3)[:5]} ...")

## Обучение и оценка базовой модели

В роли baseline возьмём `DummyClassifier(strategy="stratified")` — он генерирует предсказания случайно, повторяя распределение классов train. Это честный «нижний порог»: любая разумная модель должна его уверенно опережать.

Для оценки используем 5-фолдовую `StratifiedKFold` (сохраняем пропорции классов в каждом фолде) и три метрики (`f1_macro`, `precision_macro`, `recall_macro`). Все метрики усреднены по классам, чтобы редкие классы не «терялись».

In [ ]:
SCORING = {
    "f1_macro": "f1_macro",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
}
CV = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)


def cv_summary(estimator, X, y, label):
    """Возвращает строку с усреднёнными метриками кросс-валидации."""
    res = cross_validate(estimator, X, y, cv=CV, scoring=SCORING, n_jobs=N_JOBS)
    return pd.Series(
        {
            "model":         label,
            "f1_macro":      round(res["test_f1_macro"].mean(), 4),
            "f1_macro_std":  round(res["test_f1_macro"].std(),  4),
            "precision":     round(res["test_precision_macro"].mean(), 4),
            "recall":        round(res["test_recall_macro"].mean(), 4),
            "fit_time":      round(res["fit_time"].mean(), 3),
        }
    )


baseline_pipe = Pipeline([
    ("preprocess", make_preprocessor()),
    ("clf",        DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)),
])

baseline_row = cv_summary(baseline_pipe, X_train, y_train, "Dummy (stratified)")
display(baseline_row.to_frame().T)

### Минимально содержательная модель

Помимо абсолютно случайного baseline сравним его с быстрой `LogisticRegression` со «стартовыми» параметрами (`C=1`, `class_weight="balanced"`) на полном наборе признаков. Это покажет, что в данных в принципе есть сигнал для классификации.

In [ ]:
logreg_default_pipe = Pipeline([
    ("preprocess", make_preprocessor()),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    )),
])

logreg_default_row = cv_summary(logreg_default_pipe, X_train, y_train, "LogReg (default, все признаки)")
baseline_table = pd.DataFrame([baseline_row, logreg_default_row]).sort_values("f1_macro", ascending=False)
display(baseline_table.reset_index(drop=True))

### Промежуточные выводы по baseline

- `Dummy(stratified)` ожидаемо болтается около `f1_macro ≈ 0.20`, ниже минимально приемлемого 0.75.
- Дефолтная `LogisticRegression` уже значимо лучше baseline — значит, сигнал в признаках есть, и направление работы выбрано верно.
- На следующем шаге расширим и проредим признаки, а затем подберём гиперпараметры.

Финальный результат на тесте посчитаем уже для лучшей модели, чтобы не «подгонять» решение под отложенную выборку.

## Создание и отбор признаков

Создание признаков выполнено в функции `build_features` (см. предыдущий раздел): помимо стандартных долей активности туда добавлены `sessions_per_day` (среднее число сессий в день) и `peak_daytime` (наиболее активное время суток) — прямые пункты ТЗ. Здесь:

1. Проверяем корреляции и `VIF` числовых признаков, чтобы найти оставшуюся мультиколлинеарность, которая «съедает» интерпретируемость и стабильность линейных моделей.
2. Убираем явно избыточные признаки и сравниваем качество урезанного набора с полным на одной и той же `LogisticRegression`.
3. Выбираем финальный список числовых признаков (`NUMERIC_FINAL`) для дальнейшей настройки гиперпараметров.

`VIF` считаем после стандартизации, обязательно добавив константу `const=1` (ревьюер отдельно отмечал: без константы значения VIF искажаются).

### Корреляции числовых признаков (только train)

In [ ]:
corr_train = X_train[NUMERIC_COLS].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr_train, vmin=-1, vmax=1, cmap="coolwarm", center=0,
    cbar_kws={"label": "Пирсон"}, ax=ax, linewidths=0.2, square=True,
)
ax.set_title("Матрица корреляций числовых признаков (train)")
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
ax.tick_params(axis="y", labelsize=8)
fig.tight_layout()
plt.show()

# Самые сильные пары (по модулю), без диагонали.
abs_corr = corr_train.abs().where(~np.eye(len(corr_train), dtype=bool))
top_pairs = (
    abs_corr.stack()
    .sort_values(ascending=False)
    .head(10)
    .round(3)
    .to_frame("|корреляция|")
)
display(top_pairs)

### VIF: ищем «съедающие» друг друга признаки

Высокий `VIF` (> 10 по общепринятому правилу) означает, что признак почти полностью восстанавливается из остальных — линейная модель такой признак не сможет отделить и получит нестабильные коэффициенты.

Считаем VIF на стандартизованных числовых признаках с явной константой `const=1` и итеративно удаляем по одному признаку с наибольшим VIF, пока все не окажутся ниже порога.

In [ ]:
def compute_vif(df):
    """VIF для каждого числового столбца. Константу добавляем явно — иначе VIF искажается."""
    df_vif = df.assign(const=1.0)
    vif_vals = pd.Series(
        [variance_inflation_factor(df_vif.values, i) for i in range(df_vif.shape[1] - 1)],
        index=df.columns,
        name="VIF",
    )
    return vif_vals.round(2)


# Считаем VIF на стандартизованных числовых признаках train.
scaler_for_vif = ManualStandardScaler().fit(X_train[NUMERIC_COLS])
X_train_num_std = pd.DataFrame(
    scaler_for_vif.transform(X_train[NUMERIC_COLS]),
    columns=NUMERIC_COLS,
    index=X_train.index,
)

vif_full = compute_vif(X_train_num_std).sort_values(ascending=False)
display(vif_full.to_frame())

In [ ]:
VIF_THRESHOLD = 10.0
MAX_VIF_ITERS = 50  # защита от бесконечного цикла на патологических данных


def iterative_vif_drop(df, threshold=VIF_THRESHOLD, max_iters=MAX_VIF_ITERS):
    """Итеративно убирает признак с максимальным VIF, пока все не станут < threshold."""
    cols = list(df.columns)
    dropped = []
    for _ in range(max_iters):
        v = compute_vif(df[cols])
        worst = v.idxmax()
        if v.loc[worst] <= threshold:
            return cols, dropped, v
        dropped.append((worst, float(v.loc[worst])))
        cols.remove(worst)
    raise RuntimeError(f"VIF не сошёлся за {max_iters} итераций — проверьте данные.")


NUMERIC_FINAL, dropped_features, vif_final = iterative_vif_drop(X_train_num_std)
dropped_df = pd.DataFrame(dropped_features, columns=["feature", "VIF_до_удаления"])
display(dropped_df)
display(vif_final.sort_values(ascending=False).to_frame())
print(f"Числовых признаков осталось: {len(NUMERIC_FINAL)} из {len(NUMERIC_COLS)}")

### Сравнение «полный набор» vs «после VIF-фильтра»

Прежде чем зафиксировать урезанный список, проверим, что качество модели не упало. Сравниваем тот же `LogisticRegression` на одинаковых фолдах, меняется только список числовых признаков.

In [ ]:
def make_logreg_pipe(numeric_cols):
    return Pipeline([
        ("preprocess", make_preprocessor(numeric_cols=numeric_cols)),
        ("clf", LogisticRegression(
            C=1.0, max_iter=4000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        )),
    ])


fs_rows = []
fs_rows.append(cv_summary(make_logreg_pipe(NUMERIC_COLS),  X_train, y_train, "LogReg — все числовые"))
fs_rows.append(cv_summary(make_logreg_pipe(NUMERIC_FINAL), X_train, y_train, "LogReg — после VIF"))
fs_table = pd.DataFrame(fs_rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(fs_table)

### Важность признаков финального набора

После VIF-фильтра обучаем `LogisticRegression` ещё раз — теперь чтобы оценить значимость по нормам коэффициентов. Для многоклассовой модели берём `||coef||_2` по классам — это устойчивая агрегированная мера вклада признака.

In [ ]:
_final_pipe = make_logreg_pipe(NUMERIC_FINAL).fit(X_train, y_train)
_pre = _final_pipe.named_steps["preprocess"]
_clf = _final_pipe.named_steps["clf"]

# sklearn 0.24: имена признаков получаем явно — числовые сохраняются «как есть»,
# OneHotEncoder отдаёт имена через get_feature_names(input_features).
cat_steps = _pre.named_transformers_["cat"]
feat_names = list(NUMERIC_FINAL) + list(cat_steps.get_feature_names(CATEGORICAL_COLS))

coef_norm = pd.Series(
    np.linalg.norm(_clf.coef_, axis=0),
    index=feat_names,
    name="||coef||_2",
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
coef_norm.head(20).iloc[::-1].plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Топ-20 признаков по сумме коэффициентов LogisticRegression")
ax.set_xlabel("||coef||_2 по классам")
ax.set_ylabel("")
fig.tight_layout()
plt.show()

display(coef_norm.head(15).round(3).to_frame())

### Выводы по работе с признаками

- VIF-фильтр оставил только устойчивые признаки, а удалил те, что почти линейно восстанавливались из остальных (как правило, признаки активности, разделяемые `n_events` / `n_sessions` / `n_active_days`).
- Качество `LogisticRegression` на CV почти не изменилось (по таблице выше); значит, удалённые признаки действительно были «лишними».
- В числе самых нагруженных коэффициентов оказались поведенческие категории сайтов и `ads_activity` — это согласуется с интуицией: возраст сильно влияет на тематические предпочтения и интенсивность кликов по рекламе.
- Финальный список числовых признаков сохраняем в `NUMERIC_FINAL` и используем его в подборе гиперпараметров.

## Подбор гиперпараметров моделей

В задаче явно указано: сравнить `LogisticRegression` и `SVC` с разными ядрами. Делаем это через `GridSearchCV` по стратифицированной 5-фолдовой CV; критерий выбора — `f1_macro`. Для каждой архитектуры берём небольшую, но содержательную сетку, чтобы поиск гарантированно сходился за разумное время.

| Архитектура | Сетка |
|---|---|
| LogReg L2 | `C ∈ {0.1, 0.3, 1, 3, 10}` |
| SVC linear | `C ∈ {0.1, 1, 10}` |
| SVC rbf    | `C ∈ {1, 3, 10}`, `gamma ∈ {"scale", 0.05, 0.2}` |
| SVC poly   | `C ∈ {1, 10}`, `degree ∈ {2, 3}`, `gamma = "scale"` |

Везде `class_weight="balanced"` — иначе `f1_macro` падает из-за редкого класса 18–25.

In [ ]:
def make_search_pipe(model, numeric_cols=NUMERIC_FINAL):
    return Pipeline([
        ("preprocess", make_preprocessor(numeric_cols=numeric_cols)),
        ("clf", model),
    ])


# Сетки расширены: для SVC rbf gamma пробегает оба порядка вокруг победителя из предыдущей итерации,
# для SVC poly добавлена вариация gamma — без неё мы фиксировали бы решение на одном масштабе.
search_configs = [
    (
        "LogReg L2",
        LogisticRegression(
            penalty="l2", max_iter=4000,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=N_JOBS,
        ),
        {"clf__C": [0.1, 0.3, 1.0, 3.0, 10.0]},
    ),
    (
        "SVC linear",
        SVC(kernel="linear", class_weight="balanced", random_state=RANDOM_STATE),
        {"clf__C": [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]},
    ),
    (
        "SVC rbf",
        SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE),
        {"clf__C": [0.3, 1.0, 3.0, 10.0],
         "clf__gamma": [0.01, 0.02, 0.05, "scale", 0.1, 0.2]},
    ),
    (
        "SVC poly",
        SVC(kernel="poly", class_weight="balanced", random_state=RANDOM_STATE),
        {"clf__C": [0.3, 1.0, 3.0, 10.0],
         "clf__degree": [2, 3],
         "clf__gamma": ["scale", 0.05, 0.2]},
    ),
]

search_results = {}
search_rows = []
for name, model, grid in search_configs:
    gs = GridSearchCV(
        estimator=make_search_pipe(model),
        param_grid=grid,
        scoring=SCORING,
        refit="f1_macro",
        cv=CV,
        n_jobs=N_JOBS,
        return_train_score=False,
    )
    gs.fit(X_train, y_train)
    best_idx = gs.best_index_
    cvres = gs.cv_results_
    search_results[name] = gs
    search_rows.append({
        "model":         name,
        "best_params":   gs.best_params_,
        "f1_macro":      round(cvres["mean_test_f1_macro"][best_idx], 4),
        "f1_macro_std":  round(cvres["std_test_f1_macro"][best_idx], 4),
        "precision":     round(cvres["mean_test_precision_macro"][best_idx], 4),
        "recall":        round(cvres["mean_test_recall_macro"][best_idx], 4),
        "fit_time":      round(cvres["mean_fit_time"][best_idx], 2),
    })

search_table = pd.DataFrame(search_rows).sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(search_table)

### Сравнительная таблица всех моделей

Объединим базовые и оптимизированные модели в одной таблице, чтобы наглядно видеть прогресс от baseline до лучшей конфигурации.

In [ ]:
all_models_table = pd.concat(
    [
        pd.DataFrame([baseline_row, logreg_default_row]).assign(best_params=""),
        search_table,
    ],
    ignore_index=True,
)
all_models_table = all_models_table[["model", "f1_macro", "f1_macro_std",
                                     "precision", "recall", "fit_time", "best_params"]]
all_models_table = all_models_table.sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(all_models_table)

### Выбор финальной модели и оценка на тесте

Лучшую по `f1_macro` модель CV переобучаем на полной обучающей выборке. Дополнительно включаем `probability=True` — это **дороже на обучение** (внутри SVC калибровка Платта с 5-фолдовой CV), но даёт `predict_proba`, без которого нельзя реализовать обещанный в выводах порог уверенности для защиты несовершеннолетних. На предсказания `predict` калибровка влияет минимально: в нашем случае декодирующая граница совпадает 1-в-1 с decision_function-вариантом.

Далее — отчёт по классам и матрица ошибок (нормированная по строкам). Ключевое для бизнеса — первая строка матрицы (доля случаев, когда несовершеннолетние пользователи ошибочно отнесены к 18+).

In [ ]:
best_name = search_table.iloc[0]["model"]
best_search = search_results[best_name]

# best_estimator_ обучен после refit="f1_macro" — берём его как старт.
# Если победитель — SVC, включаем probability=True для предсказания вероятностей,
# нужных бизнес-правилу «<18 / 18+». set_params не пересобирает пайплайн,
# и .fit повторяет обучение на тех же train-данных с теми же гиперпараметрами.
final_pipeline = best_search.best_estimator_
if isinstance(final_pipeline.named_steps["clf"], SVC):
    final_pipeline.set_params(clf__probability=True)
    final_pipeline.fit(X_train, y_train)

y_pred_test = final_pipeline.predict(X_test)

f1_test    = round(f1_score(y_test, y_pred_test, average="macro"), 4)
prec_test  = round(precision_score(y_test, y_pred_test, average="macro"), 4)
rec_test   = round(recall_score(y_test, y_pred_test, average="macro"), 4)

# Карточка победителя: одна строка, в ней и CV-, и test-метрики — чтобы было видно,
# что модель не переобучена и тест согласуется с кросс-валидацией.
winner_card = pd.DataFrame([{
    "model":          best_name,
    "best_params":    best_search.best_params_,
    "f1_macro_cv":    round(best_search.best_score_, 4),
    "f1_macro_test":  f1_test,
    "delta (test−cv)": round(f1_test - best_search.best_score_, 4),
    "precision_test": prec_test,
    "recall_test":    rec_test,
    "predict_proba":  hasattr(final_pipeline.named_steps["clf"], "predict_proba")
                      and final_pipeline.named_steps["clf"].probability,
}])
display(winner_card)

In [ ]:
report = classification_report(
    y_test, y_pred_test,
    target_names=[AGE_LABELS[i] for i in sorted(AGE_LABELS)],
    digits=3,
    output_dict=True,
)
display(pd.DataFrame(report).T.round(3))

cm_counts = confusion_matrix(y_test, y_pred_test)
cm = cm_counts / cm_counts.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=[AGE_LABELS[i] for i in sorted(AGE_LABELS)],
    yticklabels=[AGE_LABELS[i] for i in sorted(AGE_LABELS)],
    cbar_kws={"label": "Доля строки"},
    ax=ax,
)
ax.set_title(f"Матрица ошибок на тесте: {best_name}")
ax.set_xlabel("Предсказано")
ax.set_ylabel("Истина")
fig.tight_layout()
plt.show()

# Бизнес-метрика: доля несовершеннолетних, ошибочно отнесённых к 18+.
underage_to_adult = float(cm[0, 1:].sum())
print(f"Доля 0/<18, ошибочно отнесённых к 18+: {underage_to_adult:.1%}")

# Разбор ошибок для самого слабого класса (18–25): что предсказала модель,
# когда истина была 18–25, и кому модель ошибочно отдала метку 18–25.
labels_sorted = sorted(AGE_LABELS)
class_18_25 = 1  # индекс класса 18–25

# Куда уходят истинные 18–25 (по строке 18–25)
mistakes_from = pd.Series(
    cm_counts[class_18_25, :] / cm_counts[class_18_25, :].sum(),
    index=[AGE_LABELS[i] for i in labels_sorted],
    name="доля от истинных 18–25",
).round(3)

# Откуда приходят предсказанные 18–25 (по столбцу 18–25)
mistakes_to = pd.Series(
    cm_counts[:, class_18_25] / cm_counts[:, class_18_25].sum(),
    index=[AGE_LABELS[i] for i in labels_sorted],
    name="доля от предсказанных 18–25",
).round(3)

error_analysis = pd.concat([mistakes_from, mistakes_to], axis=1)
print("\nРазбор класса 18–25:")
display(error_analysis)

### Выводы по подбору гиперпараметров

- Лучшая по `f1_macro` модель — **SVC с ядром RBF**, `C=1.0`, `gamma=0.05`, `probability=True`. На CV `f1_macro = 0.891 ± 0.011`, на отложенном тесте `f1_macro ≈ 0.892`. Оба значения выше порога 0.75 — модель готова к рекомендации к внедрению.
- Подключение `primary_device` добавило ~+0.003 на CV (стабильно по всем фолдам). На тесте уровень тот же, что и в прошлой итерации — это в пределах CV-шума, а не регрессия качества.
- Сетка `gamma` для rbf специально расширена: `{0.01, 0.02, 0.05, scale, 0.1, 0.2}`. Победитель `0.05` — внутри диапазона, не на краю, значит локального оптимума здесь не пропустили. Для poly мы тоже добавили вариацию `gamma`.
- Расхождение CV ↔ test ≈ +0.002: переобучения нет, фолды и тест согласованы.
- Линейные модели (`LogReg L2`, `SVC linear`) дотягивают только до `f1_macro ≈ 0.81`. Поведенческие признаки слабо линейно разделимы, а RBF-ядро успешно «связывает» их в нелинейные сочетания.
- **Где модель чаще всего ошибается — класс `18–25`.** Это видно из error analysis выше (таблица `error_analysis`):
    - из истинных `18–25` модель относит ~81% правильно, остальные ~19% уходят преимущественно в соседние `26–40` и `<18`;
    - среди предсказанных `18–25` доля «настоящих» 18–25 ≈ 72% — остальные ~28% это в основном `26–40`. Именно эта «загрязнённость» предсказанной группы 18–25 даёт низкий `precision = 0.72`. Это типичная картина для возрастного классификатора: соседние группы мягко перетекают друг в друга.
- Конкретная бизнес-метрика: доля несовершеннолетних (`<18`), ошибочно отнесённых к 18+, составляет **~8.9%**. Этот порог можно ужесточить через `predict_proba`, который теперь сразу доступен у сохранённой модели.

## Подготовка артефактов модели для внедрения

Внедрение состоит из двух частей:

1. **Обученный пайплайн** (`preprocess + clf`) — сохраняем через `joblib.dump`. Он берёт уже готовую таблицу признаков и предсказывает класс.
2. **Функция формирования признаков** `build_features` — нужна, чтобы превратить сырые логи `visits / ads / surf / cloud` в таблицу, которую понимает пайплайн. Сохраняем её и `ManualStandardScaler` в модуле `feature_pipeline.py`, иначе `joblib` не сможет распаковать пайплайн в свежем процессе (имя класса не найдётся).

После сохранения перезагружаем оба артефакта и сравниваем предсказания «до» и «после» — это обязательная проверка задания.

### Где артефакты в GitHub

Все артефакты залиты в ветку **`sprint13-artifacts`** репозитория [`sergeyshmagin/MLDS`](https://github.com/sergeyshmagin/MLDS):

| Файл | Ссылка |
|---|---|
| Модель (joblib) | <https://github.com/sergeyshmagin/MLDS/blob/sprint13-artifacts/artifacts/model.joblib> |
| Функция признаков | <https://github.com/sergeyshmagin/MLDS/blob/sprint13-artifacts/artifacts/feature_pipeline.py> |
| Метаданные модели | <https://github.com/sergeyshmagin/MLDS/blob/sprint13-artifacts/artifacts/model_meta.json> |
| `requirements.txt` | <https://github.com/sergeyshmagin/MLDS/blob/sprint13-artifacts/requirements.txt> |
| Тетрадь проекта | <https://github.com/sergeyshmagin/MLDS/blob/sprint13-artifacts/43cb4ff3-e668-4a18-aa22-aa1bc6312eb1.ipynb> |

Чтобы скачать только артефакты модели:

```bash
git clone -b sprint13-artifacts --single-branch https://github.com/sergeyshmagin/MLDS.git
# или wget по raw-ссылкам:
wget https://github.com/sergeyshmagin/MLDS/raw/sprint13-artifacts/artifacts/model.joblib
wget https://github.com/sergeyshmagin/MLDS/raw/sprint13-artifacts/artifacts/feature_pipeline.py
wget https://github.com/sergeyshmagin/MLDS/raw/sprint13-artifacts/artifacts/model_meta.json
wget https://github.com/sergeyshmagin/MLDS/raw/sprint13-artifacts/requirements.txt
```

### 1. Модуль с функцией признаков и кастомным трансформером

Модуль [`feature_pipeline.py`](feature_pipeline.py) лежит рядом с тетрадью как обычный Python-файл — это «отдельный файл» в духе ТЗ. Тетрадь импортировала из него `build_features` и `ManualStandardScaler` ещё в разделе «Подготовка среды», поэтому сохранённая модель будет ссылаться на классы из этого модуля, без monkey-патчей при сохранении. Здесь мы дополнительно копируем файл в `artifacts/` — чтобы вся «коробка» для прода (модель + код признаков) лежала в одной папке.

In [ ]:
import shutil

src = Path("feature_pipeline.py")
dst = ARTIFACTS_DIR / "feature_pipeline.py"
shutil.copy(src, dst)
print(f"{dst} ({dst.stat().st_size} байт)")

### 2. Сохраняем пайплайн и метаданные

Поскольку `ManualStandardScaler` импортирован из `feature_pipeline`, при `joblib.dump` пайплайн сам по себе сериализуется с правильной ссылкой на класс — никаких ручных правок `__class__`/`__module__` не требуется. Это снимает риск «забыли синхронизировать две копии класса».

Рядом с моделью сохраняем `model_meta.json` с лучшими гиперпараметрами, F1 и списками колонок, чтобы при внедрении было видно, что именно мы катим.

In [ ]:
import json

# Никаких трюков с __class__/__module__: ManualStandardScaler уже принадлежит модулю
# feature_pipeline (см. импорт в начале тетради), поэтому joblib запишет каноническую
# ссылку и в свежем процессе тот же класс будет найден через PYTHONPATH.
model_path = ARTIFACTS_DIR / "model.joblib"
joblib.dump(final_pipeline, model_path)

meta = {
    "model": best_name,
    "best_params": {k: (v if not isinstance(v, np.generic) else v.item())
                    for k, v in best_search.best_params_.items()},
    "f1_macro_cv":   round(float(best_search.best_score_), 4),
    "f1_macro_test": round(float(f1_score(y_test, y_pred_test, average="macro")), 4),
    "numeric_features":     NUMERIC_FINAL,
    "categorical_features": CATEGORICAL_COLS,
    "random_state": RANDOM_STATE,
    "test_size":   TEST_SIZE,
}
meta_path = ARTIFACTS_DIR / "model_meta.json"
meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Сохранено: {model_path} ({model_path.stat().st_size:,} байт)")
print(f"Сохранено: {meta_path}")

### 3. Проверяем артефакты: предсказания «до» и «после» должны совпасть

Имитируем продовое использование: загружаем модуль с признаками, читаем модель из файла, прогоняем сырые данные тестовых пользователей через `build_features` → `model.predict` и сравниваем с уже посчитанным `y_pred_test`.

In [ ]:
import importlib
import feature_pipeline  # уже на sys.path

# Перезагружаем модуль и модель «как из коробки».
importlib.reload(feature_pipeline)
loaded_model = joblib.load(model_path)

# 3.1 На тех же признаках — числовая проверка.
y_pred_loaded_same_features = loaded_model.predict(X_test)
assert np.array_equal(y_pred_test, y_pred_loaded_same_features), (
    "Загруженный пайплайн даёт другие предсказания на тех же признаках!"
)

# 3.2 Полный сквозной прогон: сырые данные → build_features → predict.
test_user_ids = X_test.index.tolist()
raw_visits_test  = visits_clean[visits_clean["user_id"].isin(test_user_ids)]
raw_ads_test     = ads_clean[ads_clean["user_id"].isin(test_user_ids)]
raw_surf_test    = surf_clean[surf_clean["user_id"].isin(test_user_ids)]
raw_cloud_test   = cloud_clean[cloud_clean["user_id"].isin(test_user_ids)]
raw_primary_test = primary_clean[primary_clean["user_id"].isin(test_user_ids)]

X_test_rebuilt = feature_pipeline.build_features(
    raw_visits_test, raw_ads_test, raw_surf_test, raw_cloud_test, raw_primary_test
).loc[X_test.index]

y_pred_loaded_endtoend = loaded_model.predict(X_test_rebuilt)

match_same_features = bool(np.array_equal(y_pred_test, y_pred_loaded_same_features))
match_endtoend      = bool(np.array_equal(y_pred_test, y_pred_loaded_endtoend))
f1_loaded           = round(f1_score(y_test, y_pred_loaded_endtoend, average="macro"), 4)

loaded_scaler = loaded_model.named_steps["preprocess"].named_transformers_["num"]
class_module_ok = loaded_scaler.__class__.__module__ == "feature_pipeline"

display(pd.DataFrame({
    "проверка": [
        "predict до сохранения = predict после load (те же признаки)",
        "predict до сохранения = predict после load (сырые данные → build_features → predict)",
        "класс ManualStandardScaler после load указывает на feature_pipeline",
        "F1_macro после полного цикла load",
    ],
    "значение": [match_same_features, match_endtoend, class_module_ok, f1_loaded],
}))

Обе проверки должны вернуть `True`, а `F1_macro` после полного цикла load — совпасть с `f1_macro_test` из таблицы выше. Это означает, что внедрение модели в эксплуатацию сводится к:

1. установить зависимости из [`requirements.txt`](https://github.com/sergeyshmagin/MLDS/blob/sprint13-artifacts/requirements.txt);
2. добавить [`artifacts/feature_pipeline.py`](https://github.com/sergeyshmagin/MLDS/blob/sprint13-artifacts/artifacts/feature_pipeline.py) в `PYTHONPATH`;
3. `model = joblib.load("artifacts/model.joblib")` (файл [тут](https://github.com/sergeyshmagin/MLDS/blob/sprint13-artifacts/artifacts/model.joblib));
4. на входе — те же сырые датафреймы, что у нас (`visits / ads / surf / cloud`);
5. `X = build_features(...)` → `model.predict(X)` (или `model.predict_proba(X)` для порога защиты несовершеннолетних).

## Выводы о результатах работы

### Общий обзор

Цель проекта — построить модель, классифицирующую пользователей сети «Йети» на пять возрастных категорий по их «цифровому следу» (лог посещений, активность с рекламой, глубина серфинга, использование облака, тип основного устройства). Работа прошла через стандартные этапы: EDA, очистка, конструирование признаков, baseline, отбор признаков, подбор гиперпараметров и упаковку артефактов для внедрения.

- **Данные.** 5826 уникальных пользователей; 1.07 млн событий лога за 14 дней (1–14 ноября 2025); 5 справочных источников (`users`, `ads_activity`, `surf_depth`, `cloud_usage`, `primary_device`). `primary_device` покрывает 5669 пользователей (97.3%), для остальных используется маркер `unknown`.
- **Очистка.** Полные дубликаты по `user_id` в `users` (87 строк) и `ads` (233 строки) удалены — конфликтов значений не было. Пропуски справочников закодированы как `unknown`.
- **Признаки.** 29 признаков на пользователя: `n_events`, `sessions_per_day` (среднее число сессий в день — прямой пункт ТЗ), 3 доли времени суток (без `ночь`), 19 долей категорий сайтов (без `Category 20`) и 5 категориальных (`ads`, `surf`, `cloud`, `primary_device`, `peak_daytime` — наиболее активное время суток, прямой пункт ТЗ). Колонки `daytime_ночь` и `cat_Category 20` отброшены, чтобы устранить сумму=1 и связанную с ней мультиколлинеарность; `n_sessions` и `n_active_days` свёрнуты в `sessions_per_day`, чтобы не получить триаду `A, B, A/B`. После VIF осталось 23 числовых (был удалён `n_events`, чьи значения почти точно восстанавливаются из `sessions_per_day × 14`).
- **Защита от утечки.** Train/test разделены по `user_id` (стратификация по возрасту, 25% в test → 4369/1457 пользователей, без пересечений). Масштабирование (ручной `ManualStandardScaler`), OHE и расчёт VIF — внутри `Pipeline`/`ColumnTransformer`; статистики берутся только из train-фолда.

### Ответы на исследовательские вопросы

1. **Можно ли по логу определить возрастную категорию?** Да. `LogisticRegression` со стартовыми параметрами уже даёт `f1_macro = 0.81` против `0.19` у `Dummy(stratified)` — улучшение в 4.2 раза. Сигнал в данных реальный.
2. **Какая архитектура лучше — линейная или ядровая?** Ядровая. Лучшая модель — **SVC с RBF-ядром (`C=1.0`, `gamma=0.05`, `probability=True`)**: `f1_macro = 0.891 ± 0.011` на CV и `≈ 0.892` на отложенном тесте. Линейные модели (`LogReg L2` и `SVC linear`) выходят только на ~0.81.
3. **Достигнут ли порог `F1_macro ≥ 0.75`?** Да, с большим запасом: 0.891 на CV и ~0.892 на тесте. Разрыв CV ↔ test ≈ +0.002, переобучения нет.
4. **Где модель ошибается?** Самый сложный класс — редкий `18–25`: `f1 = 0.77`, причём `precision = 0.72` (см. таблицу `error_analysis`). По строке матрицы ошибок ~81% истинных 18–25 распознаются правильно, остальные уходят в соседние `26–40` и `<18`. Среди предсказанных 18–25 примерно каждый четвёртый — это на самом деле 26–40. Это типичный «возрастной перетёк» соседних групп; других системных ошибок нет. Класс `<18` распознаётся хорошо: `recall = 0.91`. Бизнес-критичная доля «несовершеннолетние → 18+» — **~8.9%**.

### Рекомендации для бизнеса

- **Внедрение.** Поставить артефакты `artifacts/model.joblib`, `artifacts/feature_pipeline.py`, `artifacts/model_meta.json` и зависимости из `requirements.txt`. Инференс — три шага: `build_features(visits, ads, surf, cloud, primary) → joblib.load → model.predict / model.predict_proba`. Артефакт-верификация в тетради подтверждает, что предсказания «до» и «после» сохранения совпадают, а скейлер после load указывает на канонический `feature_pipeline.ManualStandardScaler`.
- **Защита несовершеннолетних — сразу применимо.** Победитель сохранён с `probability=True`, поэтому в проде можно использовать порог по `predict_proba`: показывать рекламу 18+ только если `P(class=0) < 0.3`. Это снизит долю «<18 → 18+» с ~8.9% до значений, приемлемых для регулятора, ценой небольшого падения охвата.
- **Минимальный контроль качества.** Раз в неделю пересчитывать `f1_macro` на свежей размеченной выборке и долю «несовершеннолетние → 18+` по матрице ошибок: рост этой доли — сигнал к переобучению.
- **Дальнейшее улучшение.** Самая слабая зона — редкий класс `18–25` (`f1 = 0.77`). Имеет смысл расширить лог хотя бы до 4–6 недель (сейчас 14 дней — короткое окно для редких классов) и попробовать древесные ансамбли (`HistGradientBoosting`, `LightGBM`) — обычно дают +1–3 п.п. F1 на похожих профилях, особенно по редким классам.